# Alpaca EDA Examples

This notebook expands on the quickstart with practical exploratory data analysis patterns for market bars, returns, volatility, correlations, intraday behavior, crypto comparison, screeners, and account context.

Create a local `.env` from `.env.example` before running the notebook. The examples only read data through `AlpacaResearchClient`; they do not place orders.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aic import AlpacaResearchClient

pd.options.display.float_format = "{:,.4f}".format
plt.style.use("seaborn-v0_8")

## Connect to Alpaca

`from_env()` searches for `.env` in the current directory and its parents, so this works when the kernel starts inside `notebooks/`.

In [ ]:
client = AlpacaResearchClient.from_env()

## Configure an EDA Window

Use a fixed historical window for reproducible examples. Change the symbols, timeframe, or date range to explore another market slice.

In [ ]:
STOCK_SYMBOLS = ["AAPL", "MSFT", "NVDA", "SPY"]
CRYPTO_SYMBOLS = ["BTC/USD", "ETH/USD"]

START = "2025-01-01"
END = None
TIMEFRAME = "1Day"

## Load and Clean Daily Stock Bars

A small preparation helper keeps timestamp handling and numeric conversion consistent across stocks, crypto, and intraday examples.

In [ ]:
def prep_bars(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    if data.empty:
        return data

    data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)
    numeric_columns = ["open", "high", "low", "close", "volume", "trade_count", "vwap"]
    for column in numeric_columns:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    data = data.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    data["date"] = data["timestamp"].dt.date
    return data


stocks = prep_bars(
    client.stock_bars(
        STOCK_SYMBOLS,
        start=START,
        end=END,
        timeframe=TIMEFRAME,
        adjustment="all",
    )
)
stocks.head()

## Inspect Schema and Missingness

Start every EDA pass with the shape, dtypes, missingness, and unique counts. These checks catch symbol coverage issues before downstream charts become misleading.

In [ ]:
schema_profile = pd.DataFrame(
    {
        "dtype": stocks.dtypes.astype(str),
        "missing": stocks.isna().sum(),
        "missing_pct": stocks.isna().mean().mul(100).round(2),
        "unique": stocks.nunique(dropna=True),
    }
).sort_index()

schema_profile

In [ ]:
coverage = (
    stocks.groupby("symbol")
    .agg(
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        rows=("close", "size"),
        avg_volume=("volume", "mean"),
        min_close=("close", "min"),
        max_close=("close", "max"),
    )
    .sort_values("rows", ascending=False)
)

coverage

## Build Price, Volume, and Return Panels

Wide panels make cross-asset analysis easier: one timestamp index, one column per symbol.

In [ ]:
price_panel = stocks.pivot(index="timestamp", columns="symbol", values="close").sort_index()
volume_panel = stocks.pivot(index="timestamp", columns="symbol", values="volume").sort_index()

daily_returns = price_panel.pct_change(fill_method=None).dropna(how="all")
log_returns = np.log(price_panel).diff().dropna(how="all")

price_panel.tail()

In [ ]:
return_summary = daily_returns.agg(["mean", "std", "min", "max"]).T
return_summary["annualized_return"] = (1 + daily_returns.mean()) ** 252 - 1
return_summary["annualized_volatility"] = daily_returns.std() * np.sqrt(252)
return_summary["sharpe_0rf"] = (
    return_summary["annualized_return"] / return_summary["annualized_volatility"]
)

return_summary.sort_values("annualized_return", ascending=False)

## Cumulative Returns and Drawdowns

Cumulative returns show growth from the start date. Drawdowns show peak-to-trough damage along the way.

In [ ]:
cumulative_returns = (1 + daily_returns.fillna(0)).cumprod().sub(1)

ax = cumulative_returns.plot(figsize=(10, 5), title="Cumulative daily returns")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Return")
plt.show()

In [ ]:
wealth_index = (1 + daily_returns.fillna(0)).cumprod()
drawdown = wealth_index.div(wealth_index.cummax()).sub(1)
max_drawdown = drawdown.min().sort_values()

ax = drawdown.plot(figsize=(10, 5), title="Drawdown from prior peak")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Drawdown")
plt.show()

max_drawdown.to_frame("max_drawdown")

## Rolling Behavior

Rolling averages and volatility help separate short-term noise from persistent shifts.

In [ ]:
ROLLING_WINDOW = 20

rolling_mean = price_panel.rolling(ROLLING_WINDOW).mean()
rolling_volatility = daily_returns.rolling(ROLLING_WINDOW).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
price_panel.plot(ax=axes[0], alpha=0.25, legend=False)
rolling_mean.plot(ax=axes[0])
axes[0].set_title(f"{ROLLING_WINDOW}-day rolling average")
axes[0].set_ylabel("Price")

rolling_volatility.plot(ax=axes[1])
axes[1].set_title(f"{ROLLING_WINDOW}-day annualized volatility")
axes[1].set_ylabel("Volatility")

fig.tight_layout()
plt.show()

## Correlation Heatmap

Correlations are regime-dependent, but a first look helps identify redundant exposures.

In [ ]:
correlation = daily_returns.corr()

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlation.columns)), labels=correlation.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlation.index)), labels=correlation.index)
ax.set_title("Daily return correlation")

for row_idx, row_label in enumerate(correlation.index):
    for col_idx, col_label in enumerate(correlation.columns):
        ax.text(col_idx, row_idx, f"{correlation.loc[row_label, col_label]:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

correlation

## Equal-Weight Portfolio Sketch

A quick benchmark-style view can reveal whether a basket behaves differently from a broad market ETF.

In [ ]:
portfolio_returns = pd.DataFrame(
    {"equal_weight": daily_returns.reindex(columns=STOCK_SYMBOLS).mean(axis=1)}
)
if "SPY" in daily_returns.columns:
    portfolio_returns["SPY"] = daily_returns["SPY"]

portfolio_growth = (1 + portfolio_returns.fillna(0)).cumprod().sub(1)
ax = portfolio_growth.plot(figsize=(10, 5), title="Equal-weight stock basket vs benchmark")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Return")
plt.show()

portfolio_returns.describe().T

## Intraday Seasonality Example

Intraday bars make it possible to inspect time-of-day volume and price behavior. Keep the date range narrow so the request stays quick.

In [ ]:
INTRADAY_SYMBOLS = ["AAPL", "MSFT"]
INTRADAY_START = "2025-04-01"
INTRADAY_END = "2025-04-15"

intraday = prep_bars(
    client.stock_bars(
        INTRADAY_SYMBOLS,
        start=INTRADAY_START,
        end=INTRADAY_END,
        timeframe="15Min",
        adjustment="all",
        limit=10_000,
    )
)

intraday["market_time"] = intraday["timestamp"].dt.tz_convert("America/New_York")
intraday["session"] = intraday["market_time"].dt.date
intraday["bar_time"] = intraday["market_time"].dt.strftime("%H:%M")
intraday["bar_return"] = intraday.groupby("symbol")["close"].pct_change(fill_method=None)

intraday.head()

In [ ]:
volume_by_time = intraday.pivot_table(
    index="bar_time", columns="symbol", values="volume", aggfunc="mean"
)
return_by_time = intraday.pivot_table(
    index="bar_time", columns="symbol", values="bar_return", aggfunc="mean"
)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
volume_by_time.plot(ax=axes[0], title="Average volume by 15-minute bar")
axes[0].set_ylabel("Volume")

return_by_time.cumsum().plot(ax=axes[1], title="Cumulative average intraday return")
axes[1].set_ylabel("Return")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()

## Crypto Comparison

Crypto trades continuously, so compare it with equities carefully. The shared date index below keeps the first pass simple.

In [ ]:
crypto = prep_bars(
    client.crypto_bars(
        CRYPTO_SYMBOLS,
        start=START,
        end=END,
        timeframe=TIMEFRAME,
    )
)

crypto_prices = crypto.pivot(index="timestamp", columns="symbol", values="close").sort_index()
crypto_returns = crypto_prices.pct_change(fill_method=None).dropna(how="all")

crypto.head()

In [ ]:
all_returns = pd.concat([daily_returns, crypto_returns], axis=1).dropna(how="all")
all_cumulative_returns = (1 + all_returns.fillna(0)).cumprod().sub(1)

ax = all_cumulative_returns.plot(figsize=(10, 5), title="Stocks and crypto cumulative returns")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Return")
plt.show()

all_returns.corr()

## Screener Context

Screeners are useful for quickly finding symbols worth a deeper follow-up EDA pass.

In [ ]:
most_active = client.most_actives(top=10)
market_movers = client.market_movers(top=10)

display(most_active)
display(market_movers)

## Account and Position Context

Paper account data can add useful portfolio context to market EDA. This section stays read-only.

In [ ]:
account = client.account()
account_columns = ["status", "currency", "buying_power", "cash", "portfolio_value"]
account.reindex(columns=[column for column in account_columns if column in account.columns])

In [ ]:
positions = client.positions()
positions_summary = positions.copy()

for column in ["qty", "market_value", "cost_basis", "unrealized_pl", "unrealized_plpc"]:
    if column in positions_summary.columns:
        positions_summary[column] = pd.to_numeric(positions_summary[column], errors="coerce")

if "market_value" in positions_summary.columns:
    positions_summary = positions_summary.sort_values("market_value", ascending=False)

positions_summary.head(20)

## Feature Table for Follow-up Analysis

EDA often ends with a tidy feature table that can feed a hypothesis test, dashboard, or model experiment.

In [ ]:
features = stocks.copy()
features["return_1d"] = features.groupby("symbol")["close"].pct_change(fill_method=None)
features["dollar_volume"] = features["close"] * features["volume"]
features["range_pct"] = (features["high"] - features["low"]) / features["close"]
features["rolling_return_5d"] = features.groupby("symbol")["return_1d"].transform(
    lambda series: series.rolling(5).mean()
)
features["rolling_vol_20d"] = features.groupby("symbol")["return_1d"].transform(
    lambda series: series.rolling(20).std() * np.sqrt(252)
)
features["volume_zscore_20d"] = features.groupby("symbol")["volume"].transform(
    lambda series: (series - series.rolling(20).mean()) / series.rolling(20).std()
)

features.dropna(subset=["return_1d"]).tail(20)

In [ ]:
eda_summary = {
    "stock_rows": len(stocks),
    "stock_symbols": sorted(stocks["symbol"].unique().tolist()),
    "first_stock_timestamp": stocks["timestamp"].min(),
    "last_stock_timestamp": stocks["timestamp"].max(),
    "crypto_rows": len(crypto),
    "crypto_symbols": sorted(crypto["symbol"].unique().tolist()),
    "feature_columns": features.columns.tolist(),
}

eda_summary